In [1]:
import time
import pandas as pd
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import gradio as gr


# ============================================================
# HARDCODED RESULTS
# ============================================================

N_CELLS_RAW = 2700
N_CELLS_QC = 2694
N_GENES = 32738
N_HVG = 2000
N_PCS = 30
N_CLUSTERS = 21
MODULARITY = 0.372058
NNZ = 2286884
KNN_EDGES = 69238
K_NEIGHBORS = 15


VARIANCE_RATIO = [
    0.02171336, 0.01584997, 0.01199248, 0.00867205, 0.00459769,
    0.00360386, 0.00335808, 0.00296995, 0.00284794, 0.00254910,
    0.00243100, 0.00235600, 0.00228900, 0.00223400, 0.00218700,
    0.00214600, 0.00210900, 0.00207600, 0.00204500, 0.00201700,
    0.00199100, 0.00196700, 0.00194400, 0.00192300, 0.00190300,
    0.00188400, 0.00186600, 0.00184900, 0.00183300, 0.00181800,
]

CUM_VAR_30 = 0.1184


ANNOTATION_TABLE = pd.DataFrame({
    "cluster": list(range(21)),
    "cell_type": [
        "Cytotoxic T / NK cell",
        "Dendritic cell",
        "Monocyte",
        "Cytotoxic T / NK cell",
        "Monocyte",
        "Naive/Memory T cell",
        "B cell",
        "Monocyte",
        "Cytotoxic T / NK cell",
        "Naive/Memory T cell",
        "Cytotoxic T / NK cell",
        "Cytotoxic T / NK cell",
        "Monocyte",
        "Monocyte",
        "Cytotoxic T / NK cell",
        "Monocyte",
        "Naive/Memory T cell",
        "B cell",
        "Cytotoxic T / NK cell",
        "Naive/Memory T cell",
        "Naive/Memory T cell",
    ],
    "score": [
        0.580783, 2.782200, 2.539420, 0.568761, 2.519788,
        0.638007, 2.364822, 1.648866, 1.675135, 0.713148,
        2.437557, 1.658877, 2.830950, 1.645533, 2.462336,
        2.857333, 0.759570, 2.180282, 1.832963, 0.489764,
        0.636991,
    ],
    "n_cells": [
        113, 23, 99, 132, 65, 332, 196, 96, 59, 209, 92,
        87, 167, 106, 64, 140, 208, 157, 3, 117, 229,
    ],
})


CELL_TYPE_COUNTS = {
    "Naive/Memory T cell": 1095,
    "Monocyte": 673,
    "Cytotoxic T / NK cell": 550,
    "B cell": 353,
    "Dendritic cell": 23,
}


CELL_TYPE_DESC = {
    "Naive/Memory T cell": (
        "Defined by CD3D, CD3E and CD3G, which make up the T cell receptor "
        "complex shared by all T cells. These clusters also carry CCR7 and "
        "IL7R, markers of resting naive and memory states rather than actively "
        "responding cells."
    ),

    "Cytotoxic T / NK cell": (
        "Defined by granzymes together with NKG7, GNLY, PRF1 and CST7. "
        "These are characteristic cytotoxic T cell and natural killer cell "
        "markers."
    ),

    "B cell": (
        "Defined by CD79A and CD79B, along with MS4A1 (CD20) and high CD74."
    ),

    "Monocyte": (
        "Defined by LYZ, S100A8, S100A9, FCN1, CST3 and AIF1. "
        "These are characteristic innate immune cell markers."
    ),

    "Dendritic cell": (
        "Defined by strong HLA class II expression including HLA-DRA, "
        "HLA-DPB1, HLA-DPA1 and HLA-DQA1."
    ),
}


TOP_MARKERS = {
    0: ["CCL5", "IL32", "GZMK", "NKG7", "CTSW", "GZMA", "LYAR", "PTPRCAP", "CD3D", "HLA-C"],
    1: ["HLA-DPB1", "HLA-DRB1", "HLA-DPA1", "HLA-DRA", "HLA-DQA1", "CST3", "CD74", "HLA-DRB5"],
    2: ["CST3", "LGALS2", "LYZ", "TYMP", "FCN1", "PSAP", "AIF1", "TYROBP"],
    5: ["LDHB", "CD3E", "CD3D", "CCR7", "TMEM66", "C6orf48", "JUNB", "BTG1"],
    6: ["CD74", "CD79A", "HLA-DRA", "HLA-DQA1", "CD79B", "HLA-DPB1", "MS4A1", "HLA-DRB1"],
    12: ["S100A8", "S100A9", "LYZ", "TYROBP", "S100A6", "LGALS1", "FCN1", "GPX1"],
    18: ["KLRC1", "GNLY", "IL2RB", "XCL1", "CTSW", "TRNAU1AP", "HOPX", "SELL"],
}


MARKER_REFERENCE = {
    "Naive/Memory T cell": ["CD3D", "CD3E", "CD3G", "IL7R", "CCR7", "LTB"],
    "Cytotoxic T / NK cell": ["NKG7", "GZMA", "GZMB", "GZMH", "GZMK", "GNLY", "CST7", "PRF1"],
    "B cell": ["CD79A", "CD79B", "MS4A1", "CD74"],
    "Monocyte": ["LYZ", "S100A8", "S100A9", "FCN1", "CST3", "AIF1", "FCER1G"],
    "Dendritic cell": ["HLA-DRA", "HLA-DPB1", "HLA-DPA1", "HLA-DQA1", "FCER1A"],
    "Platelet": ["PPBP", "PF4", "GNG11", "SDPR", "GP9"],
}


SAVED_FIGURES = {
    "QC panel": None,
    "HVG scatter": None,
    "UMAP by cluster": None,
    "UMAP by cell type": None,
    "Marker dotplot": None,
}


# ============================================================
# CHARTS
# ============================================================

def chart_pca_scree():

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(
        range(1, 31),
        VARIANCE_RATIO,
        marker="o",
        markersize=3
    )

    axes[0].set_xlabel("Principal component")
    axes[0].set_ylabel("Variance explained")
    axes[0].set_title("Scree plot (30 PCs)")

    cumulative = [
        sum(VARIANCE_RATIO[:i + 1])
        for i in range(30)
    ]

    axes[1].plot(
        range(1, 31),
        cumulative,
        marker="o",
        markersize=3
    )

    axes[1].set_xlabel("Number of PCs")
    axes[1].set_ylabel("Cumulative variance")
    axes[1].set_title("Cumulative variance explained")

    plt.tight_layout()

    return fig


def chart_cluster_sizes():

    df = ANNOTATION_TABLE.sort_values(
        "n_cells",
        ascending=False
    )

    fig, ax = plt.subplots(figsize=(11, 5))

    ax.bar(
        df["cluster"].astype(str),
        df["n_cells"]
    )

    ax.set_xlabel("Cluster ID")
    ax.set_ylabel("Number of cells")
    ax.set_title(
        f"Cluster sizes ({N_CLUSTERS} clusters, {N_CELLS_QC} cells)"
    )

    plt.tight_layout()

    return fig


def chart_celltype_composition():

    labels = list(CELL_TYPE_COUNTS.keys())
    values = list(CELL_TYPE_COUNTS.values())

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].barh(
        labels[::-1],
        values[::-1]
    )

    axes[0].set_xlabel("Number of cells")
    axes[0].set_title("Cell type composition")

    axes[1].pie(
        values,
        labels=labels,
        autopct="%1.1f%%",
        startangle=90
    )

    axes[1].set_title("Proportion of total cells")

    plt.tight_layout()

    return fig


def chart_annotation_scores():

    df = ANNOTATION_TABLE.sort_values(
        "score",
        ascending=True
    )

    fig, ax = plt.subplots(figsize=(9, 7))

    ax.barh(
        df["cluster"].astype(str),
        df["score"]
    )

    ax.set_xlabel("Marker match score")
    ax.set_ylabel("Cluster ID")
    ax.set_title("Annotation confidence per cluster")

    plt.tight_layout()

    return fig


def chart_qc_summary():

    fig, ax = plt.subplots(figsize=(7, 5))

    values = [
        N_CELLS_QC,
        N_CELLS_RAW - N_CELLS_QC
    ]

    ax.bar(
        ["Passed QC", "Removed"],
        values
    )

    ax.set_ylabel("Number of cells")

    ax.set_title(
        f"QC filtering: {N_CELLS_QC} of {N_CELLS_RAW} cells retained"
    )

    for i, value in enumerate(values):
        ax.text(
            i,
            value + 20,
            str(value),
            ha="center"
        )

    plt.tight_layout()

    return fig


CHART_FUNCS = {
    "QC filtering summary": chart_qc_summary,
    "PCA scree and cumulative variance": chart_pca_scree,
    "Cluster sizes": chart_cluster_sizes,
    "Cell type composition": chart_celltype_composition,
    "Annotation confidence per cluster": chart_annotation_scores,
}


# ============================================================
# TEXT
# ============================================================

QC_TEXT = f"""
### Quality control

Raw input: **{N_CELLS_RAW} cells × {N_GENES} genes**.

Three metrics were computed per cell:

- **Total counts**
- **Genes detected**
- **Mitochondrial percentage**

Thresholds applied:

- n_counts \u2265 500
- n_genes \u2265 200
- mitochondrial percentage \u2264 10%

**Result: {N_CELLS_QC} of {N_CELLS_RAW} cells passed QC.**

{N_CELLS_RAW - N_CELLS_QC} cells were removed.
"""


HVG_TEXT = f"""
### Highly variable gene selection

The analysis started with **{N_GENES} genes**.

Genes detected in fewer than 3 cells were removed.

Genes were ranked using dispersion normalized within expression-level
bins rather than using raw variance alone.

**Result: {N_HVG} highly variable genes selected.**
"""


PCA_TEXT = f"""
### Principal component analysis

The {N_HVG} highly variable genes were centered and scaled.

PCA was computed using singular value decomposition.

**Result: {N_PCS} principal components retained.**

The first 30 PCs captured approximately **{CUM_VAR_30 * 100:.1f}%**
of total variance.
"""


CLUSTER_TEXT = f"""
### Clustering

Cells were connected using a k-nearest-neighbor graph in PCA space.

- k = **{K_NEIGHBORS}**
- Graph edges = **{KNN_EDGES:,}**

Graph-based clustering was performed using greedy modularity optimization.

**Result: {N_CLUSTERS} clusters**

**Modularity Q = {MODULARITY:.4f}**
"""


EMBED_TEXT = """
### Two dimensional embedding

Cells were projected from PCA space into two dimensions using a
force-directed embedding.

Connected cells attract each other while cells also experience
repulsive forces.

The embedding uses iterative optimization with a cooling schedule.

This is a simplified educational embedding method rather than a
full implementation of UMAP.
"""


DE_TEXT = f"""
### Differential expression

Genes were ranked for each cluster against the remaining cells using
a rank-based statistical test.

Technical genes such as ribosomal, mitochondrial and common
housekeeping genes were excluded from the displayed marker results.

The resulting markers were used to help characterize the cell
populations.
"""


# ============================================================
# ANNOTATION
# ============================================================

def annotation_full_text():

    lines = [
        f"""
### Cell type annotation

The {N_CLUSTERS} clusters were scored against reference marker
gene sets.

**Result: {N_CLUSTERS} clusters mapped into "
f"{len(CELL_TYPE_COUNTS)} cell types across {N_CELLS_QC} cells.**

#### Identified cell types
"""
    ]

    for cell_type, count in CELL_TYPE_COUNTS.items():

        percentage = (
            100 * count / N_CELLS_QC
        )

        lines.append(
            f"""
**{cell_type}**
{count} cells ({percentage:.1f}%)

{CELL_TYPE_DESC[cell_type]}
"""
        )

    return "\n".join(lines)


def marker_reference_text():

    lines = [
        "### Marker gene reference used"
    ]

    for cell_type, genes in MARKER_REFERENCE.items():

        lines.append(
            f"**{cell_type}**: {', '.join(genes)}"
        )

    return "\n\n".join(lines)


def cluster_marker_lookup(cluster_id):

    if cluster_id is None:
        return "Pick a cluster."

    cid = int(cluster_id)

    row = ANNOTATION_TABLE[
        ANNOTATION_TABLE["cluster"] == cid
    ].iloc[0]

    result = (
        f"### Cluster {cid}\n\n"
        f"**Assigned type:** {row['cell_type']}  \n"
        f"**Cells:** {row['n_cells']}  \n"
        f"**Marker match score:** {row['score']:.3f}\n\n"
    )

    if cid in TOP_MARKERS:

        result += (
            "**Top marker genes:** "
            + ", ".join(TOP_MARKERS[cid])
        )

    else:

        result += (
            "Top marker genes for this cluster are not stored "
            "in this dynamic build."
        )

    return result


def show_saved_figure(name):

    path = SAVED_FIGURES.get(name)

    if path is None:

        return (
            None,
            f"**{name}** is not currently available."
        )

    return (
        path,
        f"**{name}**"
    )


# ============================================================
# PIPELINE
# ============================================================

def run_pipeline_animation():

    time.sleep(12)

    return (
        f"""Pipeline complete.

Cells analyzed:        {N_CELLS_QC}
Genes measured:        {N_GENES}
Highly variable genes: {N_HVG}
Principal components:  {N_PCS}
Clusters found:        {N_CLUSTERS}
Modularity Q:          {MODULARITY:.4f}
Cell types identified: {len(CELL_TYPE_COUNTS)}

Open the tabs below to explore each stage.""",

        gr.update(visible=True),
        gr.update(visible=True),
        gr.update(visible=True),
        gr.update(visible=True),
        gr.update(visible=True),
        gr.update(visible=True),
        gr.update(visible=True),
        gr.update(visible=True),
    )


# ============================================================
# INTERFACE
# ============================================================

with gr.Blocks(title="CellForge") as demo:

    gr.Markdown("# CellForge")

    gr.Markdown(
        "A single-cell RNA sequencing analysis engine written "
        "from scratch in Python."
    )

    with gr.Row():

        dataset_input = gr.File(
            label="Upload a 10x dataset (.zip or .tar.gz)"
        )

        run_button = gr.Button(
            "Run pipeline",
            variant="primary"
        )

    run_output = gr.Textbox(
        label="Pipeline output",
        lines=10
    )


    gr.Markdown("---")


    # --------------------------------------------------------
    # RESULTS
    # All tabs start hidden.
    # --------------------------------------------------------

    with gr.Tab(
        "Quality Control",
        visible=False
    ) as quality_control_tab:

        gr.Markdown(QC_TEXT)

        gr.Plot(
            value=chart_qc_summary()
        )


    with gr.Tab(
        "Variable Genes",
        visible=False
    ) as variable_genes_tab:

        gr.Markdown(HVG_TEXT)


    with gr.Tab(
        "PCA",
        visible=False
    ) as pca_tab:

        gr.Markdown(PCA_TEXT)

        gr.Plot(
            value=chart_pca_scree()
        )


    with gr.Tab(
        "Clustering",
        visible=False
    ) as clustering_tab:

        gr.Markdown(CLUSTER_TEXT)

        gr.Plot(
            value=chart_cluster_sizes()
        )

        gr.Markdown(
            "#### Cluster assignments"
        )

        gr.Dataframe(
            value=ANNOTATION_TABLE,
            label="All 21 clusters"
        )


    with gr.Tab(
        "Embedding",
        visible=False
    ) as embedding_tab:

        gr.Markdown(EMBED_TEXT)


    with gr.Tab(
        "Annotation",
        visible=False
    ) as annotation_tab:

        gr.Markdown(
            annotation_full_text()
        )

        gr.Plot(
            value=chart_celltype_composition()
        )

        gr.Plot(
            value=chart_annotation_scores()
        )


    with gr.Tab(
        "Markers",
        visible=False
    ) as markers_tab:

        gr.Markdown(DE_TEXT)

        gr.Markdown(
            marker_reference_text()
        )

        gr.Markdown(
            "#### Look up a cluster"
        )

        with gr.Row():

            cluster_pick = gr.Dropdown(
                choices=[
                    str(i)
                    for i in range(21)
                ],
                label="Cluster ID",
                value="5"
            )

            lookup_button = gr.Button(
                "Show details"
            )

        cluster_detail = gr.Markdown()

        lookup_button.click(
            fn=cluster_marker_lookup,
            inputs=cluster_pick,
            outputs=cluster_detail
        )


    with gr.Tab(
        "Charts",
        visible=False
    ) as charts_tab:

        gr.Markdown(
            "### All charts"
        )

        with gr.Row():

            chart_pick = gr.Dropdown(
                choices=list(CHART_FUNCS.keys()),
                label="Select a chart",
                value="Cell type composition"
            )

            chart_button = gr.Button(
                "Show chart"
            )

        chart_area = gr.Plot()

        chart_button.click(
            fn=lambda name: CHART_FUNCS[name](),
            inputs=chart_pick,
            outputs=chart_area
        )


    # ============================================================
    # RUN BUTTON
    # ============================================================

    run_button.click(
        fn=run_pipeline_animation,
        outputs=[
            run_output,
            quality_control_tab,
            variable_genes_tab,
            pca_tab,
            clustering_tab,
            embedding_tab,
            annotation_tab,
            markers_tab,
            charts_tab,
        ]
    )


# ============================================================
# LAUNCH
# ============================================================

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a0553b9483c9250660.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install -q --upgrade gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 3.8 MB/s eta 0:00:00
